# Edge Deployment Demo

PyTorch → ONNX → INT8 Quantization → Inference Benchmark

Run all 4 cells in order. No GPU needed.

## Cell 1: Clone + Install

In [ ]:
!cd /content && rm -rf edge-deployment-demo && git clone https://github.com/Aeijou37/edge-deployment-demo.git
%cd /content/edge-deployment-demo
!pip install onnxscript onnx onnxruntime gradio -q

## Cell 2: Export ONNX + INT8 Quantize (self-contained)

In [ ]:
import sys
sys.path.insert(0, '.')
import torch
import numpy as np
import onnx
import onnxruntime as ort
from pathlib import Path
from src.pointnet_model import PointNetClassifier
from src.quantize import quantize_onnx_int8, compare_models

model = PointNetClassifier(num_classes=40, num_points=1024)
model.eval()
print('Model created')

Path('models').mkdir(exist_ok=True)
dummy = torch.randn(1, 1024, 3)
torch.onnx.export(
    model, dummy, 'models/pointnet_fp32.onnx',
    export_params=True, opset_version=17, do_constant_folding=True,
    input_names=['input'], output_names=['logits', 'global_feat'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}, 'global_feat': {0: 'batch'}},
    dynamo=False,
)
print(f'ONNX FP32: {Path("models/pointnet_fp32.onnx").stat().st_size/1024/1024:.2f} MB')

sess = ort.InferenceSession('models/pointnet_fp32.onnx', providers=['CPUExecutionProvider'])
for i in range(5):
    x = torch.randn(1, 1024, 3)
    with torch.no_grad():
        t_out, _ = model(x)
    o_out = sess.run(['logits'], {'input': x.numpy()})[0]
    print(f'  Sample {i+1} max diff: {np.abs(t_out.numpy()-o_out).max():.6f}')
print('Verification passed')

quantize_onnx_int8('models/pointnet_fp32.onnx', 'models/pointnet_int8.onnx')
comparison = compare_models('models/pointnet_fp32.onnx', 'models/pointnet_int8.onnx', num_points=1024, num_samples=50)

## Cell 3: Benchmark

In [ ]:
from src.benchmark import benchmark_all, visualize_benchmark
from IPython.display import Image, display

results = benchmark_all(
    model, 'models/pointnet_fp32.onnx', 'models/pointnet_int8.onnx',
    num_points=1024, num_warmup=10, num_runs=50,
)
img_path = visualize_benchmark(results)
display(Image(img_path))

## Cell 4: Gradio Demo

In [ ]:
from src.app import EdgeDeployApp

app = EdgeDeployApp()
demo = app.build()
demo.launch(share=True)